In [1]:
import random
import numpy as np
import os

# 1. 파이썬 자체 난수 고정
random.seed(42)

# 2. 넘파이 난수 고정
np.random.seed(42)

# 3. 텐서플로우 사용하는 경우 (딥러닝)
import tensorflow as tf
tf.random.set_seed(42)

# 4. (선택) 실행 환경 내에서 다른 난수도 통제하고 싶을 때
os.environ['PYTHONHASHSEED'] = '42'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Conv2D, MaxPooling1D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, GridSearchCV ,cross_validate
from sklearn.metrics import *
import joblib

In [3]:
train_data = pd.read_csv('data01_train.csv')
test_data = pd.read_csv('data01_test.csv')

In [4]:
train_data.drop('subject', axis=1, inplace=True)
test_data.drop('subject', axis=1, inplace=True)

In [5]:
static_activity = (train_data['Activity']=='LAYING') | (train_data['Activity']=='SITTING') | (train_data['Activity']=='STANDING')
train_data = train_data[static_activity]

In [6]:
static_activity = (test_data['Activity']=='LAYING') | (test_data['Activity']=='SITTING') | (test_data['Activity']=='STANDING')
test_data = test_data[static_activity]

In [7]:
X_train = train_data.drop('Activity', axis=1)
y_train = train_data.loc[:, 'Activity']

In [8]:
X_train.shape

(3234, 561)

In [9]:
X_test = test_data.drop('Activity', axis=1)
y_test = test_data.loc[:, 'Activity']

In [10]:
y_train.unique()

array(['STANDING', 'LAYING', 'SITTING'], dtype=object)

In [11]:
y_test.unique()

array(['SITTING', 'STANDING', 'LAYING'], dtype=object)

In [12]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)

In [13]:
y_test_encoded = le.transform(y_test)

In [14]:
le.classes_

array(['LAYING', 'SITTING', 'STANDING'], dtype=object)

## 학습 시작

### 1차원 데이터 형태

In [15]:
cnn = Sequential()

In [16]:
cnn.add(Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(561, 1)))
cnn.add(MaxPooling1D(pool_size=2, strides=2))

cnn.add(Conv1D(filters=32, kernel_size=3, activation='relu'))
cnn.add(MaxPooling1D(pool_size=2, strides=2))

cnn.add(Flatten())
cnn.add(Dense(units=128, activation='relu'))

cnn.add(Dense(units=6, activation='softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 559, 32)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 279, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 277, 32)        │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 138, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4416)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       565,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 569,382 (2.17 MB)

 Trainable params: 569,382 (2.17 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
cnn.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [19]:
cnn.fit(X_train, y_train_encoded, batch_size=32, epochs=25)

Epoch 1/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.5675 - loss: 0.9249
Epoch 2/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.8646 - loss: 0.2927
Epoch 3/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.9276 - loss: 0.1740
Epoch 4/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.9558 - loss: 0.1271
Epoch 5/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9635 - loss: 0.1047
Epoch 6/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9677 - loss: 0.0876
Epoch 7/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9777 - loss: 0.0742
Epoch 8/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.9816 - loss: 0.0616
Epoch 9/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9850 - loss: 0.0539
Epoch 10/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9833 - loss: 0.0533
Epoch 11/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9855 - loss: 0.0482
Epoch 12/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 

In [20]:
y_pred = cnn.predict(X_test)

27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [21]:
y_pred = np.argmax(y_pred, axis=1)

In [22]:
print('accuracy :',accuracy_score(y_test_encoded, y_pred))
print('='*60)
print(confusion_matrix(y_test_encoded, y_pred))
print('='*60)
print(classification_report(y_test_encoded, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test_encoded, y_pred):.4f}")

accuracy : 0.9735894357743097
[[291   1   0]
 [  0 249   5]
 [  0  16 271]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       292
           1       0.94      0.98      0.96       254
           2       0.98      0.94      0.96       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9603


## 2차원 데이터 형태

In [23]:
X_train = X_train.values

In [24]:
X_train

array([[ 0.28850798, -0.00919614, -0.10336219, ..., -0.60111976,
         0.33129814,  0.16516332],
       [ 0.26575678, -0.01657558, -0.09816338, ...,  0.34520491,
        -0.76918611, -0.14794449],
       [ 0.27870931, -0.01451131, -0.10871733, ..., -0.83356361,
         0.20243383, -0.03275467],
       ...,
       [ 0.27719385, -0.01238895, -0.1319739 , ..., -0.91236481,
         0.11400911,  0.08014583],
       [ 0.26798135, -0.01834789, -0.10743994, ...,  0.62036305,
        -0.43724704, -0.57183996],
       [ 0.28381541, -0.04065785, -0.13801804, ..., -0.74024434,
         0.00834582, -0.14614344]])

In [25]:
X_train_reshaped = X_train.reshape(-1, 33, 17, 1)

In [26]:
X_train_reshaped.shape

(3234, 33, 17, 1)

In [27]:
cnn = Sequential()

In [28]:
cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(33, 17, 1)))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

cnn.add(Flatten())
cnn.add(Dense(units=128, activation='relu'))

cnn.add(Dense(units=6, activation='softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [29]:
cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 33, 17, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 8, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 8, 32)      │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 141,542 (552.90 KB)

 Trainable params: 141,542 (552.90 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
cnn.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [31]:
cnn.fit(X_train_reshaped, y_train_encoded, batch_size=32, epochs=25)

Epoch 1/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.4655 - loss: 1.0501
Epoch 2/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.8238 - loss: 0.3725
Epoch 3/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.8776 - loss: 0.2740
Epoch 4/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9206 - loss: 0.1863
Epoch 5/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.9514 - loss: 0.1376
Epoch 6/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9614 - loss: 0.1120
Epoch 7/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9677 - loss: 0.0948
Epoch 8/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9726 - loss: 0.0828
Epoch 9/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9769 - loss: 0.0720
Epoch 10/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.9802 - loss: 0.0637
Epoch 11/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9824 - loss: 0.0570
Epoch 12/25
102/102 ━━━━━━━━━━━━━━━━━━━━ 

In [32]:
X_test.shape

(833, 561)

In [33]:
X_test = X_test.values

In [34]:
X_test_reshaped = X_test.reshape(-1, 33, 17, 1)

In [35]:
X_test_reshaped.shape

(833, 33, 17, 1)

In [36]:
y_pred = cnn.predict(X_test_reshaped)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


In [37]:
y_pred = np.argmax(y_pred, axis=1)

In [38]:
print('accuracy :',accuracy_score(y_test_encoded, y_pred))
print('='*60)
print(confusion_matrix(y_test_encoded, y_pred))
print('='*60)
print(classification_report(y_test_encoded, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test_encoded, y_pred):.4f}")

accuracy : 0.9471788715486195
[[292   0   0]
 [  0 254   0]
 [  0  44 243]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       292
           1       0.85      1.00      0.92       254
           2       1.00      0.85      0.92       287

    accuracy                           0.95       833
   macro avg       0.95      0.95      0.95       833
weighted avg       0.95      0.95      0.95       833

Cohen's Kappa Score: 0.9209
